In [5]:
import hist
import gzip
import uproot
import correctionlib
import correctionlib.schemav2 as cs
import matplotlib.pyplot as plt
from pathlib import Path

In [6]:
data_path = Path.cwd().parent / "analysis/data/hzz_lepton_sf"

electron_files = {
    "2022preEE": {
        "id": f"{data_path}/SF2022eleID_preEE.root",
        "reco_low": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_2022_rerecoBCD_rerecoE_PromptFG_RMS_30052022/lowpT/Run3_2022BCD_New_lowpT_mergeEta/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_mid": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/New_SF_19122023/midpT/Run3_2022BCD_New_midpT7/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_high": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/New_SF_19122023/highpT/Run3_2022BCD_New_highpt6/passingRECO/egammaEffi.txt_EGM2D.root", 
    },
    "2022postEE": {
        "id": f"{data_path}/SF2022eleID_postEE.root",
        "reco_low": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_2022_rerecoBCD_rerecoE_PromptFG_RMS_30052022/lowpT/Run3_2022BCD_New_lowpT_mergeEta/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_mid": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/New_SF_19122023/midpT/Run3_2022BCD_New_midpT7/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_high": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/New_SF_19122023/highpT/Run3_2022BCD_New_highpt6/passingRECO/egammaEffi.txt_EGM2D.root", 
    },
    "2023preBPix": {
        "id": f"{data_path}/SF2023eleID_preBPix.root",
        "id_gap": f"{data_path}/SF2023eleID_Gap_preBPix.root",
        "reco_low": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/lowpT/Run3_2023C_New_lowpT_mergeEta_Added_symmetrizationsystEta_29052024/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_mid": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/midpT/Run3_2023C_New_midpT2_eta/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_high": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/highpT/Run3_2023C_New_highpt1_eta/passingRECO/egammaEffi.txt_EGM2D.root",
    },
    "2023postBPix": {
        "id": f"{data_path}/SF2023eleID_postBPix.root",
        "id_gap": f"{data_path}/SF2023eleID_Gap_preBPix.root",
        "id_hole": f"{data_path}/SF2023eleID_postBPix_Hole.root",
        "reco_low": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/lowpT/Run3_2023D_New_lowpT_mergeEta_Added_symmetrizationsystEta_29052024/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_mid": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/midpT/Run3_2023D_New_midpT_eta2/passingRECO/egammaEffi.txt_EGM2D.root",
        "reco_high": "/eos/cms/store/group/phys_egamma/validation/web/Run3_egm_reco_SF/SF_prompt_2023_19012024/highpT/Run3_2023D_New_highpt_eta2/passingRECO/egammaEffi.txt_EGM2D.root", 
    },
    "2024": {
        "id": f"{data_path}/SF2024eleID.root",
        "id_gap": f"{data_path}/SF2024eleID_Gap.root",
        "reco_low": "/eos/cms/store/group/phys_egamma/ScaleFactors/Data2024/EleReco/lowPt/egammaEffi.txt_EGM2D.root",
        "reco_mid": "/eos/cms/store/group/phys_egamma/ScaleFactors/Data2024/EleReco/midPt/egammaEffi.txt_EGM2D.root",
        "reco_high": "/eos/cms/store/group/phys_egamma/ScaleFactors/Data2024/EleReco/highPt/egammaEffi.txt_EGM2D.root",
    }
}

In [7]:
# Read the four electron SF histograms and their bin uncertainties.
electron_hists = {}

for correction_name in ["id", "reco_low", "reco_mid", "reco_high"]:
    with uproot.open(electron_files["2022preEE"][correction_name]) as f:
        root_hist = f["EGamma_SF2D"]

        electron_hists[correction_name] = {
            "edges": [
                root_hist.axes[0].edges().tolist(),
                root_hist.axes[1].edges().tolist(),
            ],
            "nominal": root_hist.values(flow=False),
            "error": root_hist.errors(flow=False),
        }


# Convert a 2D array into a correctionlib MultiBinning node.
def array_to_multibinning(edges, values):
    return cs.MultiBinning(
        nodetype="multibinning",
        inputs=["etaSC", "pt"],
        edges=edges,
        content=values.ravel(order="C").tolist(),
        flow="clamp",
    )


# Build the correction categories.
correction_categories = []

for correction_name, hist_data in electron_hists.items():

    variation_categories = [
        cs.CategoryItem(
            key=variation,
            value=array_to_multibinning(
                hist_data["edges"],
                hist_data[variation],
            ),
        )
        for variation in ["nominal", "error"]
    ]

    correction_categories.append(
        cs.CategoryItem(
            key=correction_name,
            value=cs.Category(
                nodetype="category",
                input="variation",
                content=variation_categories,
            ),
        )
    )


# Define a single correction for all electron SFs and uncertainties.
electron_sf = cs.Correction(
    name="electron_sf",
    description="2022 preEE electron ID and reconstruction scale factors",
    version=1,
    inputs=[
        cs.Variable(
            name="etaSC",
            type="real",
            description="Electron supercluster pseudorapidity",
        ),
        cs.Variable(
            name="pt",
            type="real",
            description="Electron transverse momentum",
        ),
        cs.Variable(
            name="correction",
            type="string",
            description="Correction type",
        ),
        cs.Variable(
            name="variation",
            type="string",
            description="Nominal scale factor or absolute uncertainty",
        ),
    ],
    output=cs.Variable(
        name="weight",
        type="real",
        description="Electron scale factor or absolute uncertainty",
    ),
    data=cs.Category(
        nodetype="category",
        input="correction",
        content=correction_categories,
    ),
)


# Create and save the CorrectionSet.
cset = cs.CorrectionSet(
    schema_version=2,
    description="Run 3 electron scale factors and uncertainties",
    corrections=[electron_sf],
)

output_file = str(data_path / "electron_sf_2022preEE.json.gz")

with gzip.open(output_file, "wt", encoding="utf-8") as f:
    f.write(cset.model_dump_json(exclude_unset=True, indent=2))


print(f"\nCorrection saved to: {output_file}")


Correction saved to: /afs/cern.ch/user/d/daocampo/public/higgscharm/analysis/data/hzz_lepton_sf/electron_sf_2022preEE.json.gz


In [8]:
# Read the four electron SF histograms and their bin uncertainties.
electron_hists = {}

for correction_name in ["id", "reco_low", "reco_mid", "reco_high"]:
    with uproot.open(electron_files["2022postEE"][correction_name]) as f:
        root_hist = f["EGamma_SF2D"]

        electron_hists[correction_name] = {
            "edges": [
                root_hist.axes[0].edges().tolist(),
                root_hist.axes[1].edges().tolist(),
            ],
            "nominal": root_hist.values(flow=False),
            "error": root_hist.errors(flow=False),
        }


# Convert a 2D array into a correctionlib MultiBinning node.
def array_to_multibinning(edges, values):
    return cs.MultiBinning(
        nodetype="multibinning",
        inputs=["etaSC", "pt"],
        edges=edges,
        content=values.ravel(order="C").tolist(),
        flow="clamp",
    )


# Build the correction categories.
correction_categories = []

for correction_name, hist_data in electron_hists.items():

    variation_categories = [
        cs.CategoryItem(
            key=variation,
            value=array_to_multibinning(
                hist_data["edges"],
                hist_data[variation],
            ),
        )
        for variation in ["nominal", "error"]
    ]

    correction_categories.append(
        cs.CategoryItem(
            key=correction_name,
            value=cs.Category(
                nodetype="category",
                input="variation",
                content=variation_categories,
            ),
        )
    )


# Define a single correction for all electron SFs and uncertainties.
electron_sf = cs.Correction(
    name="electron_sf",
    description="2022 preEE electron ID and reconstruction scale factors",
    version=1,
    inputs=[
        cs.Variable(
            name="etaSC",
            type="real",
            description="Electron supercluster pseudorapidity",
        ),
        cs.Variable(
            name="pt",
            type="real",
            description="Electron transverse momentum",
        ),
        cs.Variable(
            name="correction",
            type="string",
            description="Correction type",
        ),
        cs.Variable(
            name="variation",
            type="string",
            description="Nominal scale factor or absolute uncertainty",
        ),
    ],
    output=cs.Variable(
        name="weight",
        type="real",
        description="Electron scale factor or absolute uncertainty",
    ),
    data=cs.Category(
        nodetype="category",
        input="correction",
        content=correction_categories,
    ),
)


# Create and save the CorrectionSet.
cset = cs.CorrectionSet(
    schema_version=2,
    description="Run 3 electron scale factors and uncertainties",
    corrections=[electron_sf],
)

output_file = str(data_path / "electron_sf_2022postEE.json.gz")

with gzip.open(output_file, "wt", encoding="utf-8") as f:
    f.write(cset.model_dump_json(exclude_unset=True, indent=2))


print(f"\nCorrection saved to: {output_file}")


Correction saved to: /afs/cern.ch/user/d/daocampo/public/higgscharm/analysis/data/hzz_lepton_sf/electron_sf_2022postEE.json.gz


In [9]:
# Read the electron SF histograms and their bin uncertainties.
electron_hists = {}

for correction_name in [
    "id",
    "id_gap",
    "reco_low",
    "reco_mid",
    "reco_high",
]:
    with uproot.open(electron_files["2023preBPix"][correction_name]) as f:
        root_hist = f["EGamma_SF2D"]

        electron_hists[correction_name] = {
            "edges": [
                root_hist.axes[0].edges().tolist(),
                root_hist.axes[1].edges().tolist(),
            ],
            "nominal": root_hist.values(flow=False),
            "error": root_hist.errors(flow=False),
        }


# Convert a 2D array into a correctionlib MultiBinning node.
def array_to_multibinning(edges, values):
    return cs.MultiBinning(
        nodetype="multibinning",
        inputs=["etaSC", "pt"],
        edges=edges,
        content=values.ravel(order="C").tolist(),
        flow="clamp",
    )


# Build the correction categories.
correction_categories = []

for correction_name, hist_data in electron_hists.items():

    variation_categories = [
        cs.CategoryItem(
            key=variation,
            value=array_to_multibinning(
                hist_data["edges"],
                hist_data[variation],
            ),
        )
        for variation in ["nominal", "error"]
    ]

    correction_categories.append(
        cs.CategoryItem(
            key=correction_name,
            value=cs.Category(
                nodetype="category",
                input="variation",
                content=variation_categories,
            ),
        )
    )


# Define a single correction for all electron SFs and uncertainties.
electron_sf = cs.Correction(
    name="electron_sf",
    description="2023 preBPix electron ID and reconstruction scale factors",
    version=1,
    inputs=[
        cs.Variable(
            name="etaSC",
            type="real",
            description="Electron supercluster pseudorapidity",
        ),
        cs.Variable(
            name="pt",
            type="real",
            description="Electron transverse momentum",
        ),
        cs.Variable(
            name="correction",
            type="string",
            description="Correction type",
        ),
        cs.Variable(
            name="variation",
            type="string",
            description="Nominal scale factor or absolute uncertainty",
        ),
    ],
    output=cs.Variable(
        name="weight",
        type="real",
        description="Electron scale factor or absolute uncertainty",
    ),
    data=cs.Category(
        nodetype="category",
        input="correction",
        content=correction_categories,
    ),
)


# Create and save the CorrectionSet.
cset = cs.CorrectionSet(
    schema_version=2,
    description="Run 3 electron scale factors and uncertainties",
    corrections=[electron_sf],
)

output_file = str(data_path / "electron_sf_2023preBPix.json.gz")

with gzip.open(output_file, "wt", encoding="utf-8") as f:
    f.write(cset.model_dump_json(exclude_unset=True, indent=2))


print(f"\nCorrection saved to: {output_file}")


Correction saved to: /afs/cern.ch/user/d/daocampo/public/higgscharm/analysis/data/hzz_lepton_sf/electron_sf_2023preBPix.json.gz


In [10]:
# Read the electron SF histograms and their bin uncertainties.
electron_hists = {}

for correction_name in [
    "id",
    "id_gap",
    "id_hole",
    "reco_low",
    "reco_mid",
    "reco_high",
]:
    with uproot.open(electron_files["2023postBPix"][correction_name]) as f:
        root_hist = f["EGamma_SF2D"]

        electron_hists[correction_name] = {
            "edges": [
                root_hist.axes[0].edges().tolist(),
                root_hist.axes[1].edges().tolist(),
            ],
            "nominal": root_hist.values(flow=False),
            "error": root_hist.errors(flow=False),
        }


# Convert a 2D array into a correctionlib MultiBinning node.
def array_to_multibinning(edges, values):
    return cs.MultiBinning(
        nodetype="multibinning",
        inputs=["etaSC", "pt"],
        edges=edges,
        content=values.ravel(order="C").tolist(),
        flow="clamp",
    )


# Build the correction categories.
correction_categories = []

for correction_name, hist_data in electron_hists.items():

    variation_categories = [
        cs.CategoryItem(
            key=variation,
            value=array_to_multibinning(
                hist_data["edges"],
                hist_data[variation],
            ),
        )
        for variation in ["nominal", "error"]
    ]

    correction_categories.append(
        cs.CategoryItem(
            key=correction_name,
            value=cs.Category(
                nodetype="category",
                input="variation",
                content=variation_categories,
            ),
        )
    )


# Define a single correction for all electron SFs and uncertainties.
electron_sf = cs.Correction(
    name="electron_sf",
    description="2023 postBPix electron ID and reconstruction scale factors",
    version=1,
    inputs=[
        cs.Variable(
            name="etaSC",
            type="real",
            description="Electron supercluster pseudorapidity",
        ),
        cs.Variable(
            name="pt",
            type="real",
            description="Electron transverse momentum",
        ),
        cs.Variable(
            name="correction",
            type="string",
            description="Correction type",
        ),
        cs.Variable(
            name="variation",
            type="string",
            description="Nominal scale factor or absolute uncertainty",
        ),
    ],
    output=cs.Variable(
        name="weight",
        type="real",
        description="Electron scale factor or absolute uncertainty",
    ),
    data=cs.Category(
        nodetype="category",
        input="correction",
        content=correction_categories,
    ),
)


# Create and save the CorrectionSet.
cset = cs.CorrectionSet(
    schema_version=2,
    description="Run 3 electron scale factors and uncertainties",
    corrections=[electron_sf],
)

output_file = str(data_path / "electron_sf_2023postBPix.json.gz")

with gzip.open(output_file, "wt", encoding="utf-8") as f:
    f.write(cset.model_dump_json(exclude_unset=True, indent=2))


print(f"\nCorrection saved to: {output_file}")


Correction saved to: /afs/cern.ch/user/d/daocampo/public/higgscharm/analysis/data/hzz_lepton_sf/electron_sf_2023postBPix.json.gz


In [11]:
# Read the electron SF histograms and their bin uncertainties.
electron_hists = {}

for correction_name in [
    "id",
    "id_gap",
    "reco_low",
    "reco_mid",
    "reco_high",
]:
    with uproot.open(electron_files["2024"][correction_name]) as f:
        root_hist = f["EGamma_SF2D"]

        electron_hists[correction_name] = {
            "edges": [
                root_hist.axes[0].edges().tolist(),
                root_hist.axes[1].edges().tolist(),
            ],
            "nominal": root_hist.values(flow=False),
            "error": root_hist.errors(flow=False),
        }


# Convert a 2D array into a correctionlib MultiBinning node.
def array_to_multibinning(edges, values):
    return cs.MultiBinning(
        nodetype="multibinning",
        inputs=["etaSC", "pt"],
        edges=edges,
        content=values.ravel(order="C").tolist(),
        flow="clamp",
    )


# Build the correction categories.
correction_categories = []

for correction_name, hist_data in electron_hists.items():

    variation_categories = [
        cs.CategoryItem(
            key=variation,
            value=array_to_multibinning(
                hist_data["edges"],
                hist_data[variation],
            ),
        )
        for variation in ["nominal", "error"]
    ]

    correction_categories.append(
        cs.CategoryItem(
            key=correction_name,
            value=cs.Category(
                nodetype="category",
                input="variation",
                content=variation_categories,
            ),
        )
    )


# Define a single correction for all electron SFs and uncertainties.
electron_sf = cs.Correction(
    name="electron_sf",
    description="2024 preBPix electron ID and reconstruction scale factors",
    version=1,
    inputs=[
        cs.Variable(
            name="etaSC",
            type="real",
            description="Electron supercluster pseudorapidity",
        ),
        cs.Variable(
            name="pt",
            type="real",
            description="Electron transverse momentum",
        ),
        cs.Variable(
            name="correction",
            type="string",
            description="Correction type",
        ),
        cs.Variable(
            name="variation",
            type="string",
            description="Nominal scale factor or absolute uncertainty",
        ),
    ],
    output=cs.Variable(
        name="weight",
        type="real",
        description="Electron scale factor or absolute uncertainty",
    ),
    data=cs.Category(
        nodetype="category",
        input="correction",
        content=correction_categories,
    ),
)


# Create and save the CorrectionSet.
cset = cs.CorrectionSet(
    schema_version=2,
    description="Run 3 electron scale factors and uncertainties",
    corrections=[electron_sf],
)

output_file = str(data_path / "electron_sf_2024.json.gz")

with gzip.open(output_file, "wt", encoding="utf-8") as f:
    f.write(cset.model_dump_json(exclude_unset=True, indent=2))


print(f"\nCorrection saved to: {output_file}")


Correction saved to: /afs/cern.ch/user/d/daocampo/public/higgscharm/analysis/data/hzz_lepton_sf/electron_sf_2024.json.gz
